# Data Preparation Template

Load data from shared path, run basic checks, split dataset, and save lightweight metadata.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
try:
    import yaml
except ImportError as e:
    raise ImportError("PyYAML is required. Run: pip install pyyaml") from e

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CFG_LOCAL = REPO_ROOT / 'config' / 'paths.local.yaml'
CFG_TEMPLATE = REPO_ROOT / 'config' / 'paths.template.yaml'
CFG_PATH = CFG_LOCAL if CFG_LOCAL.exists() else CFG_TEMPLATE

with open(CFG_PATH, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

INPUT_PATH = Path(cfg['shared']['modeling_input'])
METADATA_DIR = REPO_ROOT / cfg['repo']['metadata_dir']
for d in [METADATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Input:', INPUT_PATH)
print('Metadata dir:', METADATA_DIR)

In [ ]:
# TODO: update these
LABEL_COL = 'label'
READ_NROWS = None
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALID_SIZE = 0.2
IMBALANCE_PREP_MODE = None  # None | 'downsample_majority'

In [ ]:
suffix = INPUT_PATH.suffix.lower()
if suffix == '.tsv':
    df = pd.read_csv(INPUT_PATH, sep='\t', nrows=READ_NROWS)
elif suffix == '.csv':
    df = pd.read_csv(INPUT_PATH, nrows=READ_NROWS)
elif suffix == '.parquet':
    df = pd.read_parquet(INPUT_PATH)
    if READ_NROWS is not None:
        df = df.head(READ_NROWS)
else:
    raise ValueError(f'Unsupported file type: {suffix}')

if LABEL_COL not in df.columns:
    raise KeyError(f'Missing LABEL_COL: {LABEL_COL}')

print('Shape:', df.shape)
display(df.head(3))

In [ ]:
null_summary = df.isna().sum().sort_values(ascending=False).to_frame('null_count')
null_summary['null_ratio'] = null_summary['null_count'] / len(df)
label_dist = df[LABEL_COL].value_counts(dropna=False).to_frame('count')
label_dist['ratio'] = label_dist['count'] / label_dist['count'].sum()
display(null_summary.head(20))
display(label_dist)

In [ ]:
work_df = df.copy()
if IMBALANCE_PREP_MODE == 'downsample_majority':
    min_count = work_df[LABEL_COL].value_counts().min()
    work_df = (
        work_df.groupby(LABEL_COL, group_keys=False)
        .apply(lambda x: x.sample(n=min_count, random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
    print('Downsampled shape:', work_df.shape)
else:
    print('No prep-stage sampling.')

train_val_df, test_df = train_test_split(
    work_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=work_df[LABEL_COL]
)
train_df, valid_df = train_test_split(
    train_val_df,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=train_val_df[LABEL_COL]
)
print('train:', train_df.shape, 'valid:', valid_df.shape, 'test:', test_df.shape)

In [ ]:
split_summary = pd.DataFrame({
    'split': ['train', 'valid', 'test'],
    'rows': [len(train_df), len(valid_df), len(test_df)]
})
split_summary.to_csv(METADATA_DIR / 'split_summary.csv', index=False)
label_dist.to_csv(METADATA_DIR / 'label_distribution_overall.csv')
null_summary.reset_index().rename(columns={'index': 'column'}).to_csv(METADATA_DIR / 'null_summary.csv', index=False)

profile = {
    'input_path': str(INPUT_PATH),
    'label_col': LABEL_COL,
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'valid_size': VALID_SIZE,
    'imbalance_prep_mode': IMBALANCE_PREP_MODE
}
with open(METADATA_DIR / 'data_profile.json', 'w', encoding='utf-8') as f:
    json.dump(profile, f, indent=2)

print('Saved metadata files to', METADATA_DIR)